#  OpenGrid — GRPO Training Notebook

**Multi-Agent RL for Power Grid Operations**

This notebook trains an LLM (Qwen 2.5 1.5B) to operate a power grid using GRPO (Group Relative Policy Optimization).

- **Environment**: OpenGrid — multi-agent POMDP with safety layer & oversight agent
- **Task**: Maintain 50 Hz frequency, prevent line overloads, avoid blackouts
- **Training**: TRL GRPOTrainer + Unsloth 4-bit quantization

 **Runtime**: Select `T4 GPU` from Runtime → Change runtime type

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install fastapi uvicorn pydantic numpy networkx matplotlib openai httpx datasets

## 2. Clone OpenGrid Repository

In [ ]:
import os

#  UPDATE THIS with your actual repo URL
REPO_URL = "https://github.com/krishnagoyal099/Opengrid_env.git"

if not os.path.exists("opengrid"):
    !git clone {REPO_URL} opengrid
else:
    !cd opengrid && git pull

os.chdir("opengrid")
print(f"Working directory: {os.getcwd()}")
!ls -la

## 3. Verify GPU & Environment

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(" No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Verify OpenGrid imports work
import sys
sys.path.insert(0, '.')

from src.environment import OpenGridEnv
from src.tasks import TASKS
from src.models import GridAction, BusAdjustment

print(f"Available tasks: {list(TASKS.keys())}")
for tid, cfg in TASKS.items():
    print(f"  {tid}: {cfg['num_buses']} buses, {cfg['num_agents']} agents, {cfg.get('difficulty','')}")

## 4. Run Test Mode (Pipeline Verification)

In [ ]:
!python training/train_grpo.py --test-mode

## 5. Baseline Evaluation (Before Training)

Run the heuristic policy to get baseline scores. We'll compare against this after training.

In [ ]:
import json
import re
import numpy as np
from src.environment import OpenGridEnv
from src.tasks import TASKS
from src.models import GridAction, BusAdjustment
from training.train_grpo import (
    rollout_multi_agent, format_observation_prompt, extract_action
)

def heuristic_generate(prompt):
    """Simple proportional controller as baseline."""
    freq_match = re.search(r'Frequency: ([\d.]+)', prompt)
    freq = float(freq_match.group(1)) if freq_match else 50.0
    error = 50.0 - freq
    delta = max(-20, min(20, error * 10))
    bus_match = re.search(r'Bus (\d+) \((generator|battery|slack)\)', prompt)
    if bus_match:
        return json.dumps({"bus_adjustments": [{"bus_id": int(bus_match.group(1)), "delta": round(delta, 1)}], "topology_actions": []})
    return json.dumps({"bus_adjustments": [], "topology_actions": []})

# Evaluate baseline on all tasks
baseline_results = {}
for task_id in ["task_easy", "task_medium", "task_karnataka"]:
    if task_id not in TASKS:
        continue
    config = TASKS[task_id]
    rewards = []
    import copy
    for ep in range(5):
        ep_config = copy.deepcopy(config)
        ep_config['seed'] = 42 + ep
        env = OpenGridEnv(ep_config)
        result = rollout_multi_agent(env, heuristic_generate, ep_config)
        rewards.append(result['total_reward'])
    baseline_results[task_id] = {
        "avg_reward": np.mean(rewards),
        "std_reward": np.std(rewards),
        "rewards": rewards
    }
    print(f"[BASELINE] {task_id}: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}")

# Save baseline for later comparison
import pickle
os.makedirs("training/outputs", exist_ok=True)
with open("training/outputs/baseline_results.pkl", "wb") as f:
    pickle.dump(baseline_results, f)
print("\n Baseline scores saved.")

## 6. Load Model with Unsloth (4-bit Quantized)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f" Model loaded: {MODEL_NAME}")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Generate Training Prompts from Environment

In [ ]:
import copy
import json as _json
import numpy as np
from training.train_grpo import SYSTEM_PROMPT, format_observation_prompt

TRAIN_TASK = "task_karnataka"  # Change to task_easy for faster first run
NUM_EPISODES = 30

task_config = TASKS[TRAIN_TASK]
base_seed = task_config.get('seed', 42)

prompts = []
obs_contexts = []  # stored as JSON strings to satisfy PyArrow schema inference

for episode in range(NUM_EPISODES):
    ep_config = copy.deepcopy(task_config)
    ep_config['seed'] = base_seed + episode
    env = OpenGridEnv(ep_config)
    zone_obs = env.reset_multi()

    for t in range(min(10, task_config['max_steps'])):
        for agent_id, obs in zone_obs.items():
            # model_dump_json() → json.loads() ensures all keys are strings
            obs_dict = _json.loads(obs.model_dump_json())
            prompt_text = format_observation_prompt(obs_dict, zone_name=obs.zone_name)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt_text},
            ]
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompts.append(formatted)
            # Store as JSON string — flat scalar, no schema-inference issues
            obs_contexts.append(_json.dumps(obs_dict))

        # Advance env with diverse random actions (no slack bus)
        random_actions = {}
        for aid in range(env.num_agents):
            zone_buses = task_config['zone_bus_ids'].get(aid, [])
            controllable = [bid for bid in zone_buses
                if next((b for b in task_config['buses'] if b['id'] == bid), {}).get('type')
                in ['generator', 'battery']]
            adj = []
            if controllable:
                bid = np.random.choice(controllable)
                adj = [BusAdjustment(bus_id=int(bid), delta=float(np.random.uniform(-15, 15)))]
            random_actions[aid] = GridAction(bus_adjustments=adj)

        result = env.step_multi(random_actions)
        if result.done:
            break
        zone_obs = result.observations

print(f" Generated {len(prompts)} training prompts")
print(f"\nSample prompt (first 400 chars):")
print(prompts[0][:400])

## 8. Define GRPO Reward Function

In [ ]:
import json as _json
from training.train_grpo import compute_grpo_reward_env, extract_action

def reward_fn(completions, obs_context=None, **kwargs):
    """GRPO reward function with env-grounded physics rewards."""
    texts = []
    for c in completions:
        if isinstance(c, list):
            text = c[-1]["content"] if c else ""
        else:
            text = str(c)
        texts.append(text)

    if obs_context is None:
        batch_obs = [None] * len(texts)
    else:
        batch_obs = [
            _json.loads(ctx) if isinstance(ctx, str) else ctx
            for ctx in obs_context
        ]
    return compute_grpo_reward_env(texts, batch_obs, task_config, horizon=3)

# Sanity test
test_rewards = reward_fn([
    '{"bus_adjustments": [{"bus_id": 1, "delta": 5.0}], "topology_actions": []}',
    "invalid json here",
])
print(f"Test rewards: {test_rewards}")
assert len(test_rewards) == 2
print("[OK] reward_fn works")


## 9. Train with GRPO 

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

_cuda_ok = torch.cuda.is_available()
_bf16   = _cuda_ok and torch.cuda.is_bf16_supported()
_fp16   = _cuda_ok and not _bf16

grpo_config = GRPOConfig(
    output_dir="training/outputs/grpo_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
    max_completion_length=256,
    num_generations=8,
    report_to="none",
    remove_unused_columns=False,
    bf16=_bf16,
    fp16=_fp16,
)

# obs_contexts are JSON strings — PyArrow handles flat strings with no issues
train_dataset = Dataset.from_dict({"prompt": prompts, "obs_context": obs_contexts})
print(f"Dataset: {len(train_dataset)} rows, columns: {train_dataset.column_names}")

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print(f"Training on {len(prompts)} prompts, {grpo_config.num_train_epochs} epoch(s)")
print(f"Effective batch size: {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")
print("\n Starting GRPO training...")

train_result = trainer.train()

print("\n Training complete!")
print(f"   Total steps: {trainer.state.global_step}")

## 10. Save Trained Model

In [ ]:
OUTPUT_PATH = "training/outputs/trained_model"
trainer.save_model(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f" Model saved to {OUTPUT_PATH}")

## 11. Evaluate Trained Model (After Training)

In [ ]:
from transformers import pipeline

# Create generation function from trained model
FastLanguageModel.for_inference(model)

def trained_generate(prompt):
    """Generate action using the trained model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# Evaluate on same tasks as baseline
trained_results = {}
for task_id in ["task_easy", "task_medium", "task_karnataka"]:
    if task_id not in TASKS:
        continue
    config = TASKS[task_id]
    rewards = []
    import copy
    for ep in range(5):
        ep_config = copy.deepcopy(config)
        ep_config['seed'] = 42 + ep
        env = OpenGridEnv(ep_config)
        result = rollout_multi_agent(env, trained_generate, ep_config)
        rewards.append(result['total_reward'])
        print(f"  {task_id} ep{ep}: reward={result['total_reward']:.2f}, blackout={result['is_blackout']}")
    trained_results[task_id] = {
        "avg_reward": np.mean(rewards),
        "std_reward": np.std(rewards),
        "rewards": rewards
    }
    print(f"[TRAINED] {task_id}: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}\n")

## 12. Generate Before/After Plots 

In [ ]:
import matplotlib.pyplot as plt
import pickle

# Load baseline
with open("training/outputs/baseline_results.pkl", "rb") as f:
    baseline_results = pickle.load(f)

# ── Plot 1: Before vs After Bar Chart ──
common_tasks = [t for t in baseline_results if t in trained_results]
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(common_tasks))
width = 0.35

before_vals = [baseline_results[t]['avg_reward'] for t in common_tasks]
after_vals  = [trained_results[t]['avg_reward']  for t in common_tasks]

bars1 = ax.bar(x - width/2, before_vals, width, label='Heuristic Baseline', color='#ff6b6b', alpha=0.8)
bars2 = ax.bar(x + width/2, after_vals,  width, label='GRPO Trained',       color='#00d4aa', alpha=0.8)

ax.set_xlabel('Task', fontsize=12)
ax.set_ylabel('Average Episode Reward', fontsize=12)
ax.set_title('OpenGrid — GRPO Training: Before vs After', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t.replace('task_', '').title() for t in common_tasks])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Fix label positioning for negative bar heights
for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.,
            h + (2 if h >= 0 else -5),
            f'{h:.1f}',
            ha='center', va='bottom' if h >= 0 else 'top', fontsize=10
        )

plt.tight_layout()
plt.savefig('training/outputs/before_after.png', dpi=150)
plt.show()
print(" Saved: training/outputs/before_after.png")

In [ ]:
# ── Plot 2: Training Reward Curve ──
history = trainer.state.log_history

steps = [h['step'] for h in history if 'loss' in h]
losses = [h['loss'] for h in history if 'loss' in h]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps, losses, color='#ff6b6b', linewidth=1.5, alpha=0.6, label='Loss')
if len(losses) > 10:
    window = min(20, len(losses) // 3)
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    ax.plot(steps[window-1:], smoothed, color='#ff6b6b', linewidth=2.5, label=f'Smoothed (w={window})')

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('OpenGrid GRPO — Training Loss', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training/outputs/training_loss.png', dpi=150)
plt.show()
print(" Saved: training/outputs/training_loss.png")

## 13. Summary & Next Steps

### Results Table

In [ ]:
print("="*60)
print("  OpenGrid GRPO Training — Results Summary")
print("="*60)

# Rebuild common_tasks in case Cell 12 was skipped
common_tasks = [t for t in baseline_results if t in trained_results]

print(f"{'Task':<20} {'Baseline':>12} {'Trained':>12} {'Δ':>10}")
print("-"*60)
for t in common_tasks:
    b = baseline_results[t]['avg_reward']
    a = trained_results[t]['avg_reward']
    delta = a - b
    arrow = '↑' if delta > 0 else '↓'
    print(f"{t:<20} {b:>10.2f}   {a:>10.2f}   {arrow} {abs(delta):.2f}")
print("="*60)

In [ ]:
# Display plots inline
from IPython.display import Image, display
display(Image("training/outputs/before_after.png"))
display(Image("training/outputs/training_loss.png"))
